# Predictive accuracy: full observations

Manuscript Section 6.2 / Table 2 and supplementary model comparison.

Use the full-data fits with partially observed station–period vectors. Pairwise and GH08 use nearest-PSD covariance correction followed by 0.01 I. IOX uses its native covariance without a nugget or repair. Other models retain the original 1e-7 training shrinkage and 1e-6 jitter.

The full evaluation contains **134 events and 110,225 observed residuals**. Parameters stay fixed. Entry and station holdouts mask 10% and use 20 seed-0 repetitions; period holdouts mask each of the nine periods once. Every model sees the same masks.

The notebook contains the recorded all-event results from the supplied fit, an executed whole-event demonstration, and the command for rerunning the complete evaluation. It writes no result files unless an `output` directory is explicitly passed to `evaluate`.


In [1]:
from pathlib import Path
import sys
import hashlib
from io import StringIO
import pandas as pd
from IPython.display import display
from threadpoolctl import threadpool_limits

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'scripts/models').is_dir())
sys.path.insert(0, str(ROOT))
from scripts.analysis.predictive_accuracy.evaluate import evaluate, smallest_event
pd.set_option('display.max_columns', 20)
pd.set_option('display.precision', 6)

DATASET = 'full'
CACHE = ROOT / 'results/analysis/model_fit/models.pkl'

## Recorded all-event evaluation

These values were computed from the supplied model file in the completed evaluation. The hash identifies that fit. RMSE and CRPS in the main table pool squared errors, CRPS sums and observation counts across the held-out residuals. The following cell also shows per-repeat/period means and win counts, making the aggregation explicit.

In [2]:
# Recorded all-event evaluation, 11 September 2026, before reorganizing the repository.
# These CSV strings preserve the completed numerical results inside the notebook.
# They are not results of the short demonstration below.
RECORDED_MODELS_SHA256 = 'ad4d35c1b0a9281a0b4e1f86bcac23891a6f26bf3886f0ca400434a9f39e75ac'
recorded_summary = pd.read_csv(StringIO('method,wls,log_likelihood,pseudo_rmse,pseudo_crps,station_rmse,station_crps,period_rmse,period_crps\r\nIndependent,,-164732.19545065443,1.086990476100426,0.6022944003340152,1.0837067714053126,0.6015918078716113,1.0833455258884692,0.6004475423862807\r\nPairwise empirical semivariogram,0.1220831002667234,-67787.6500351611,0.4533484384223757,0.2104066427178627,1.0569385540116276,0.55172485884855,0.4753081950659282,0.2355434335278244\r\nGH08 semivariogram,0.2082588774631403,-67469.50175769524,0.3856874981608756,0.1973012163571218,0.9198250481444514,0.4982409331946497,0.4258001387413292,0.2120168196826831\r\nKronecker semivariogram,0.3446503833419919,-62245.12496435039,0.3642702060806414,0.1811406352001496,0.8871113406689697,0.4823886109421139,0.3909327937743447,0.1914261057931647\r\nPCA MLE,1.9366685271150788,-64359.577651220345,0.3649822110674167,0.1796760418638423,0.875967671109599,0.4764846706137991,0.3929779593429547,0.1910300449021464\r\nPCA semivariogram,1.1973830075642318,-65838.50373295123,0.3680977784851805,0.1816641377464572,0.8795979275210689,0.4778974799264365,0.3976739324797871,0.1936469393348467\r\nLMC block MLE,0.2214594370358612,-58091.72815332357,0.3545491266043644,0.1761656574868446,0.874480314720028,0.474978072134094,0.3850885455123096,0.1891875494866505\r\nLMC semivariogram,0.1614752938250442,-59531.09031635008,0.3592582814625886,0.1788751631371381,0.8755715947067749,0.4757821653126937,0.3920131804703304,0.1944470435961701\r\nMultivariate Matern semivariogram,0.1022477890158247,-61298.24210095508,0.3663191029894615,0.1821554559845135,0.893620809740904,0.4849165494323079,0.4004546889061006,0.1975928319397625\r\nSBSS semivariogram,1.0976055253302628,-61921.40342898903,0.3642819732596237,0.1795709205797316,0.8827498412897348,0.4795434481117556,0.3926983253530217,0.1914473534778912\r\nIOX full semivariogram,0.1482866226894109,-65620.95568279154,0.3719049254073843,0.1882598745685505,0.8963792751278539,0.4873143743106879,0.4057232016551214,0.2039566531858815\r\n'))
recorded_statistics = pd.read_csv(StringIO('method,task,metric,mean,sd,se,wins,n,mean_rank\r\nGH08 semivariogram,random_holdout,rmse,0.385676901434804,0.0029332822004109,0.0006559018397309,0,20,9.0\r\nIOX full semivariogram,random_holdout,rmse,0.3718901739815521,0.0033984575824937,0.0007599182173105,0,20,8.0\r\nKronecker semivariogram,random_holdout,rmse,0.3642578072425261,0.0030835558591433,0.0006895040513462,0,20,3.8\r\nLMC block MLE,random_holdout,rmse,0.3545340682731032,0.0033525496409324,0.0007496528895067,20,20,1.0\r\nLMC semivariogram,random_holdout,rmse,0.359244101753822,0.0032748088802599,0.0007322695269581,0,20,2.0\r\nMultivariate Matern semivariogram,random_holdout,rmse,0.3663066430125166,0.0030998301998419,0.0006931431045553,0,20,5.95\r\nPCA MLE,random_holdout,rmse,0.3649689360496767,0.003193762741629,0.0007141470594288,0,20,4.75\r\nPCA semivariogram,random_holdout,rmse,0.3680837478830384,0.0032973785602137,0.0007373162608188,0,20,6.95\r\nPairwise empirical semivariogram,random_holdout,rmse,0.4533209126747862,0.0051254526583524,0.0011460860559533,0,20,10.0\r\nSBSS semivariogram,random_holdout,rmse,0.364267610368341,0.0033188581523488,0.0007421192436331,0,20,3.55\r\nGH08 semivariogram,random_holdout,mean_crps,0.1973012163571217,0.0013274890532536,0.0002968355762461,0,20,9.0\r\nIOX full semivariogram,random_holdout,mean_crps,0.1882598745685505,0.0014587252071652,0.0003261808723714,0,20,8.0\r\nKronecker semivariogram,random_holdout,mean_crps,0.1811406352001496,0.0013922704144096,0.0003113211289681,0,20,5.2\r\nLMC block MLE,random_holdout,mean_crps,0.1761656574868447,0.0015025241090987,0.0003359746045777,20,20,1.0\r\nLMC semivariogram,random_holdout,mean_crps,0.1788751631371381,0.0014169293140746,0.0003168350265583,0,20,2.0\r\nMultivariate Matern semivariogram,random_holdout,mean_crps,0.1821554559845135,0.0013567897921006,0.0003033874206314,0,20,6.85\r\nPCA MLE,random_holdout,mean_crps,0.1796760418638423,0.0014772017749697,0.0003303123585315,0,20,3.55\r\nPCA semivariogram,random_holdout,mean_crps,0.1816641377464573,0.0015028927112842,0.000336057026532,0,20,5.95\r\nPairwise empirical semivariogram,random_holdout,mean_crps,0.2104066427178627,0.0013949914686778,0.0003119295751995,0,20,10.0\r\nSBSS semivariogram,random_holdout,mean_crps,0.1795709205797316,0.0014763731322117,0.0003301270683779,0,20,3.45\r\nGH08 semivariogram,station_holdout,rmse,0.9196632466579324,0.0175182064019839,0.0039171900358708,0,20,9.0\r\nIOX full semivariogram,station_holdout,rmse,0.8961910523137234,0.0186964017177567,0.0041806425175547,0,20,7.85\r\nKronecker semivariogram,station_holdout,rmse,0.8869126971046851,0.0190788937605535,0.0042661703384094,0,20,6.0\r\nLMC block MLE,station_holdout,rmse,0.8742635672774866,0.019829570470272,0.004434026753615,11,20,1.45\r\nLMC semivariogram,station_holdout,rmse,0.8753526607636379,0.0199554804882995,0.0044621810895508,3,20,2.25\r\nMultivariate Matern semivariogram,station_holdout,rmse,0.8934311943807829,0.0187297806765945,0.0041881062796527,0,20,7.15\r\nPCA MLE,station_holdout,rmse,0.875739513792445,0.0203427749157027,0.0045487827562488,6,20,2.3\r\nPCA semivariogram,station_holdout,rmse,0.8793860341514257,0.0196453350195007,0.004392830454436,0,20,4.0\r\nPairwise empirical semivariogram,station_holdout,rmse,1.056712060636846,0.0225161541284342,0.0050347651223041,0,20,10.0\r\nSBSS semivariogram,station_holdout,rmse,0.8825421066538413,0.0194816113236096,0.004356220723082,0,20,5.0\r\nGH08 semivariogram,station_holdout,mean_crps,0.4982394576647105,0.007532189247051,0.0016842487175799,0,20,9.0\r\nIOX full semivariogram,station_holdout,mean_crps,0.4873131784823504,0.0076128505568893,0.0017022851347751,0,20,7.95\r\nKronecker semivariogram,station_holdout,mean_crps,0.4823869701140787,0.0076556967792215,0.0017118658413465,0,20,6.0\r\nLMC block MLE,station_holdout,mean_crps,0.4749769768391265,0.0080596348769721,0.0018021891458737,16,20,1.2\r\nLMC semivariogram,station_holdout,mean_crps,0.4757812248959714,0.0081312526548822,0.0018182033678542,1,20,2.25\r\nMultivariate Matern semivariogram,station_holdout,mean_crps,0.484915260558644,0.0076940620907187,0.0017204445857951,0,20,7.05\r\nPCA MLE,station_holdout,mean_crps,0.4764831052076991,0.008576966725577,0.0019178680639144,3,20,2.55\r\nPCA semivariogram,station_holdout,mean_crps,0.4778959457200635,0.008176079343277,0.0018282269200999,0,20,4.0\r\nPairwise empirical semivariogram,station_holdout,mean_crps,0.5517246107134547,0.0095603590092089,0.0021377612633893,0,20,10.0\r\nSBSS semivariogram,station_holdout,mean_crps,0.4795419384908842,0.0079130561893237,0.0017694131549103,0,20,5.0\r\nGH08 semivariogram,period_holdout,rmse,0.4240793290395431,0.2224516803656217,,0,9,8.333333333333334\r\nIOX full semivariogram,period_holdout,rmse,0.4027656738560576,0.1965862341103159,,0,9,8.11111111111111\r\nKronecker semivariogram,period_holdout,rmse,0.3845890875771361,0.2026892803293799,,0,9,3.111111111111111\r\nLMC block MLE,period_holdout,rmse,0.3790778073828902,0.1986514166417609,,9,9,1.0\r\nLMC semivariogram,period_holdout,rmse,0.3854744432905842,0.2023505775788394,,0,9,3.6666666666666665\r\nMultivariate Matern semivariogram,period_holdout,rmse,0.3956481497215245,0.2048991865882729,,0,9,6.666666666666667\r\nPCA MLE,period_holdout,rmse,0.3866868620470757,0.1990941556026964,,0,9,4.222222222222222\r\nPCA semivariogram,period_holdout,rmse,0.3903645091254748,0.2005491947768689,,0,9,5.666666666666667\r\nPairwise empirical semivariogram,period_holdout,rmse,0.4698878959134291,0.2509230654865468,,0,9,9.77777777777778\r\nSBSS semivariogram,period_holdout,rmse,0.3865459903585406,0.1987296213122581,,0,9,4.444444444444445\r\nGH08 semivariogram,period_holdout,mean_crps,0.2357757998032462,0.1276649912666339,,0,9,8.333333333333334\r\nIOX full semivariogram,period_holdout,mean_crps,0.2233272166340328,0.1116498269861087,,0,9,8.222222222222221\r\nKronecker semivariogram,period_holdout,mean_crps,0.2115480990394782,0.1176968952716942,,0,9,3.4444444444444446\r\nLMC block MLE,period_holdout,mean_crps,0.2089000264335447,0.1152410144903782,,6,9,2.0\r\nLMC semivariogram,period_holdout,mean_crps,0.2144372579151508,0.1171049015614797,,0,9,5.777777777777778\r\nMultivariate Matern semivariogram,period_holdout,mean_crps,0.2182565824287737,0.1188116722235283,,0,9,6.444444444444445\r\nPCA MLE,period_holdout,mean_crps,0.2107198499822443,0.117848880317985,,3,9,2.6666666666666665\r\nPCA semivariogram,period_holdout,mean_crps,0.2130871427571709,0.118322309450962,,0,9,4.666666666666667\r\nPairwise empirical semivariogram,period_holdout,mean_crps,0.2620384574588361,0.1454886182542311,,0,9,9.77777777777778\r\nSBSS semivariogram,period_holdout,mean_crps,0.211062806287469,0.1172829408216926,,0,9,3.6666666666666665\r\n'))
display(pd.DataFrame([{'models_file': CACHE.name,
    'recorded_models_sha256': RECORDED_MODELS_SHA256,
    'current_file_matches_recorded_fit': hashlib.sha256(CACHE.read_bytes()).hexdigest() == RECORDED_MODELS_SHA256,
    'events': 134, 'observations': 110225}]))
display(recorded_summary)


,models_file,recorded_models_sha256,current_file_matches_recorded_fit,events,observations
0,models.pkl,ad4d35c1b0a9281a0b4e1f86bcac23891a6f26bf3886f0...,True,134,110225


,method,wls,log_likelihood,pseudo_rmse,pseudo_crps,station_rmse,station_crps,period_rmse,period_crps
0,Independent,NaN,-164732.195451,1.086990,0.602294,1.083707,0.601592,1.083346,0.600448
1,Pairwise empirical semivariogram,0.122083,-67787.650035,0.453348,0.210407,1.056939,0.551725,0.475308,0.235543
2,GH08 semivariogram,0.208259,-67469.501758,0.385687,0.197301,0.919825,0.498241,0.425800,0.212017
3,Kronecker semivariogram,0.344650,-62245.124964,0.364270,0.181141,0.887111,0.482389,0.390933,0.191426
4,PCA MLE,1.936669,-64359.577651,0.364982,0.179676,0.875968,0.476485,0.392978,0.191030
5,PCA semivariogram,1.197383,-65838.503733,0.368098,0.181664,0.879598,0.477897,0.397674,0.193647
6,LMC block MLE,0.221459,-58091.728153,0.354549,0.176166,0.874480,0.474978,0.385089,0.189188
7,LMC semivariogram,0.161475,-59531.090316,0.359258,0.178875,0.875572,0.475782,0.392013,0.194447
8,Multivariate Matern semivariogram,0.102248,-61298.242101,0.366319,0.182155,0.893621,0.484917,0.400455,0.197593
9,SBSS semivariogram,1.097606,-61921.403429,0.364282,0.179571,0.882750,0.479543,0.392698,0.191447


In [3]:
display(recorded_statistics.pivot(index='method', columns=['task', 'metric'], values='wins'))
display(recorded_statistics[['method', 'task', 'metric', 'mean', 'sd', 'n']])

task                              random_holdout           station_holdout  \
metric                                      rmse mean_crps            rmse   
method                                                                       
GH08 semivariogram                             0         0               0   
IOX full semivariogram                         0         0               0   
Kronecker semivariogram                        0         0               0   
LMC block MLE                                 20        20              11   
LMC semivariogram                              0         0               3   
Multivariate Matern semivariogram              0         0               0   
PCA MLE                                        0         0               6   
PCA semivariogram                              0         0               0   
Pairwise empirical semivariogram               0         0               0   
SBSS semivariogram                             0         0               0   

task                                        period_holdout            
metric                            mean_crps           rmse mean_crps  
method                                                                
GH08 semivariogram                        0              0         0  
IOX full semivariogram                    0              0         0  
Kronecker semivariogram                   0              0         0  
LMC block MLE                            16              9         6  
LMC semivariogram                         1              0         0  
Multivariate Matern semivariogram         0              0         0  
PCA MLE                                   3              0         3  
PCA semivariogram                         0              0         0  
Pairwise empirical semivariogram          0              0         0  
SBSS semivariogram                        0              0         0

,method,task,metric,mean,sd,n
0,GH08 semivariogram,random_holdout,rmse,0.385677,0.002933,20
1,IOX full semivariogram,random_holdout,rmse,0.371890,0.003398,20
2,Kronecker semivariogram,random_holdout,rmse,0.364258,0.003084,20
3,LMC block MLE,random_holdout,rmse,0.354534,0.003353,20
4,LMC semivariogram,random_holdout,rmse,0.359244,0.003275,20
5,Multivariate Matern semivariogram,random_holdout,rmse,0.366307,0.003100,20
6,PCA MLE,random_holdout,rmse,0.364969,0.003194,20
7,PCA semivariogram,random_holdout,rmse,0.368084,0.003297,20
8,Pairwise empirical semivariogram,random_holdout,rmse,0.453321,0.005125,20
9,SBSS semivariogram,random_holdout,rmse,0.364268,0.003319,20


## Run a complete real event

This cell actually recomputes all models on the smallest whole event, with 20 entry holdouts, 20 station holdouts, and each available period held out in turn. Masks are generated over the entire dataset before selecting this event, so they match the full evaluation exactly. The displayed likelihood and prediction scores are for this event only; WLS is a fit diagnostic on the entire fitted dataset.

In [4]:
event_id = smallest_event(CACHE, DATASET)
with threadpool_limits(limits=1):
    example = evaluate(CACHE, dataset=DATASET, event_ids=[event_id])
print(f'Live evaluation: event {event_id}')
display(example['run'])
display(example['summary'])

Live evaluation: event 1087


,dataset,seed,repeats,events,observations,mask_sha256,models_sha256,smoke,whole_dataset
0,full,0,20,1,240,842d490d60fb8223013c81da86b0dae07d11a9f191cfe1...,ad4d35c1b0a9281a0b4e1f86bcac23891a6f26bf3886f0...,False,False


,method,wls,log_likelihood,pseudo_rmse,pseudo_crps,station_rmse,station_crps,period_rmse,period_crps
0,Independent,NaN,-348.664802,1.012896,0.572655,0.976724,0.555624,1.035845,0.582494
1,Pairwise empirical semivariogram,0.122083,-76.688332,0.252780,0.128732,0.944640,0.528500,0.267342,0.133086
2,GH08 semivariogram,0.208259,-69.977385,0.251109,0.126556,0.935615,0.524072,0.275106,0.135870
3,Kronecker semivariogram,0.344650,-45.185713,0.246622,0.117572,0.949516,0.530093,0.270716,0.128829
4,PCA MLE,1.936669,-35.474732,0.239306,0.114201,0.963303,0.540149,0.254474,0.120634
5,PCA semivariogram,1.197383,-34.103080,0.241090,0.114593,0.951983,0.532425,0.253257,0.119269
6,LMC block MLE,0.221459,-53.470017,0.249799,0.121022,0.944044,0.522637,0.267019,0.127628
7,LMC semivariogram,0.161475,-60.801418,0.247074,0.121671,0.941482,0.521721,0.265563,0.133445
8,Multivariate Matern semivariogram,0.102248,-55.658107,0.247534,0.121014,0.945511,0.528706,0.271292,0.132358
9,SBSS semivariogram,1.097606,-44.813799,0.243299,0.118560,0.951281,0.530151,0.259177,0.124551


## Rerun every event

Set `RUN_ALL_EVENTS = True` and execute this cell to rebuild the full table and replicate statistics from the model PKL. This is computationally expensive because each event requires a joint covariance factorization for each model. The results remain in memory and are displayed directly below. The two input PKLs are not modified.

In [5]:
RUN_ALL_EVENTS = False
if RUN_ALL_EVENTS:
    with threadpool_limits(limits=1):
        full_run = evaluate(CACHE, dataset=DATASET, verbose=True)
    display(full_run['summary'])
    display(full_run['replicate_statistics'])
else:
    print('Full rerun not requested. The recorded all-event table and live single-event example are shown above.')

Full rerun not requested. The recorded all-event table and live single-event example are shown above.


## Inspect individual predictions or export a table

`example` (or `full_run`) contains `event_likelihood`, `random_holdout`, `station_holdout`, `period_holdout`, `summary`, `replicate_statistics`, and `run` DataFrames. Gaussian conditioning is implemented by `ConditionalCache` in `evaluate.py`. To export a table when needed, call its standard `to_csv` method.

In [6]:
display(example['random_holdout'][['method', 'eqid', 'repeat', 'n_observations', 'sse', 'crps_sum']].head())
# Optional: example['summary'].to_csv('event_summary.csv', index=False)

,method,eqid,repeat,n_observations,sse,crps_sum
0,Independent,1087,0,24,26.943765,13.902988
1,Independent,1087,1,24,27.079436,14.298965
2,Independent,1087,2,24,33.929738,16.478007
3,Independent,1087,3,24,30.751323,15.523076
4,Independent,1087,4,24,23.568476,13.249196
